In [17]:
# Imports
import os
import cv2
from tqdm import tqdm
from pprint import pprint as print

In [19]:
ORIGINAL_EGO4D_PATH = "/u/dduka/work/projects/AVION/datasets/Ego4D/ego4d/v2/full_scale" 
EGO4D_PATH = "/u/dduka/work/projects/AVION/datasets/Ego4D/video_320px_15sec"

In [25]:
# Compute expected number of chuncks per video
chunk_length = 15
expected_chunks_per_video = {}

for mp4_file in tqdm(os.listdir(ORIGINAL_EGO4D_PATH)):
    try:
        video_path = os.path.join(ORIGINAL_EGO4D_PATH, mp4_file)
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps
        num_chunks = int(duration // chunk_length)
        expected_chunks_per_video[mp4_file] = num_chunks
    except Exception as e:
        print(f"Error processing {mp4_file}: {e}")

  0%|          | 0/9822 [00:00<?, ?it/s]

 36%|███▌      | 3525/9822 [10:53<17:33,  5.98it/s]

'Error processing manifest.csv: float division by zero'


100%|██████████| 9822/9822 [30:47<00:00,  5.32it/s]


In [26]:
video_count = {}

# Iterate over the directories inside
for root, dirs, files in os.walk(EGO4D_PATH):
    mp4_files = [f for f in files if f.lower().endswith(".mp4")]
    if mp4_files:
        video_count[root.split("/")[-1]] = len(mp4_files)

# Summary for dirs
print(f"Total directories: {len(video_count)}")
print(video_count)
print(video_count["0ad80ef0-076c-40cb-852a-ac36360d377a.mp4"])

'Total directories: 9821'
{'000786a7-3f9d-4fe6-bfb3-045b368f7d44.mp4': 28,
 '000a3525-6c98-4650-aaab-be7d2c7b9402.mp4': 2,
 '000cd456-ff8d-499b-b0c1-4acead128a8b.mp4': 64,
 '001e3e4e-2743-47fc-8564-d5efd11f9e90.mp4': 9,
 '00277df3-9107-4592-ba85-b8d054149551.mp4': 7,
 '002ad105-bd9a-4858-953e-54e88dc7587e.mp4': 71,
 '002c3b5c-ed86-4af3-99a1-4b497b7c8a86.mp4': 184,
 '002d2729-df71-438d-8396-5895b349e8fd.mp4': 239,
 '0030b1e9-c6a6-4809-a495-8d45791f9775.mp4': 11,
 '0031d268-818c-4ec4-a804-935be610a61a.mp4': 133,
 '003a90b1-7960-44e2-ba2f-308ba83adc70.mp4': 8,
 '003b145d-42d3-470d-b987-8a489c42f2f8.mp4': 8,
 '0049fdd8-0044-4ef5-9c34-b3469416ebe5.mp4': 91,
 '004a1802-c546-4dcc-86ba-bf1080077017.mp4': 97,
 '0051a756-bce3-4cf2-adb2-3c4b5cde6711.mp4': 13,
 '0066ab25-04ad-41b2-89ab-283d2bfa1c4b.mp4': 38,
 '0075dfd4-03b2-48c3-bde1-d7851bc8e367.mp4': 81,
 '007cb0df-4f4f-4810-b246-8ba6639f53e1.mp4': 37,
 '0089fd8c-1b30-468e-b0c9-98d265e4d705.mp4': 4,
 '0090c729-7bdd-4f99-8a76-1df2c7990ab1.mp4': 1

In [ ]:
corrupted_videos = []

# Iterate over the directories inside
for root, dirs, files in tqdm(os.walk(EGO4D_PATH)):
    mp4_files = [f for f in files if f.lower().endswith(".mp4")]

    # Open the file to check if it is valid
    for mp4_file in mp4_files:
        cap = cv2.VideoCapture(os.path.join(root, mp4_file))
        if not cap.isOpened():
            print(f"Cannot open video file: {mp4_file}")
            corrupted_videos.append(mp4_file)

# Summary for dirs
print(f"Total corrupted videos: {len(corrupted_videos)}")
print(corrupted_videos)

In [29]:
frame_ids = [78520, 78529, 78535, 78541]
rel_frame_ids__ = [220, 229, 235, 241]
vr_len = 449
chunk = 2610
chunk_len = 15

print(f"int(chunk * fps): {int(chunk * fps)}")
print(f"int((chunk + chunk_len) * fps): {int((chunk + chunk_len) * fps)}")
print(f"frame_ids: {frame_ids}")

rel_frame_ids = list(
    filter(
        lambda x: int(chunk * fps) <= x < int((chunk + chunk_len) * fps),
        frame_ids,
    )
)
rel_frame_ids = [int(frame_id - chunk * fps) for frame_id in rel_frame_ids]
rel_frame_ids

'int(chunk * fps): 78300'
'int((chunk + chunk_len) * fps): 78750'
'frame_ids: [78520, 78529, 78535, 78541]'


[220, 229, 235, 241]

In [104]:
import numpy as np

frame_ids = [6743, 6746, 6749, 6752]
rel_frame_ids__ = [443, 446, 449]
vr_len = 449
chunk = 210
chunk_len = 15

print(f"int(chunk * fps): {int(chunk * fps)}")
print(f"int((chunk + chunk_len) * fps): {int((chunk + chunk_len) * fps)}")
print(f"frame_ids: {frame_ids}")

rel_frame_ids = list(
    filter(
        lambda x: int(chunk * fps) <= x < int((chunk + chunk_len) * fps - 1),
        frame_ids,
    )
)

rel_frame_ids = [int(frame_id - chunk * fps) for frame_id in rel_frame_ids]
print(rel_frame_ids)


def get_frame_ids(start_frame, end_frame, num_segments=32, jitter=True):
    frame_ids = np.convolve(
        np.linspace(start_frame, end_frame, num_segments + 1), [0.5, 0.5], mode="valid"
    )
    frame_ids = np.linspace(start_frame, end_frame, num_segments + 1)
    
    if jitter:
        seg_size = float(end_frame - start_frame - 1) / num_segments
        shift = (np.random.rand(num_segments) - 0.5) * seg_size
        frame_ids += shift
    return frame_ids.astype(int).tolist()

get_frame_ids(7360, 7384, num_segments=4, jitter=False)


'int(chunk * fps): 6300'
'int((chunk + chunk_len) * fps): 6750'
'frame_ids: [6743, 6746, 6749, 6752]'
[443, 446]


[7360, 7366, 7372, 7378, 7384]